# Multi-Model Probability Fast Pipeline

This version keeps the original modeling flow but removes plotting and consolidates loading, nodata handling, derived-field setup, ecohydrology, and landslide probability into a scenario-ready pipeline.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from landlab.components import FlowAccumulator, SinkFillerBarnes
from landlab.components.landslides import LandslideProbability
from landlab.grid.mappers import map_node_to_cell
from landlab.io import esri_ascii

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "notebook").exists():
    REPO_ROOT = REPO_ROOT.parent
NOTEBOOK_DIR = REPO_ROOT / "notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from potential_evapotranspiration_field_OFFICIAL import PotentialEvapotranspiration
from radiation_field_OFFICIAL import Radiation
from soil_moisture_dynamics import SoilMoisture

SCENARIO_DIR = Path("/mnt/c/Users/amehedi/Downloads/ml_debris/pioneer/output/cut1")
DEFAULT_FORCING_DIR = REPO_ROOT / "data" / "forcing"

PRECIP_CSV = SCENARIO_DIR / "precip_2026-01.csv"
TEMP_CSV = SCENARIO_DIR / "temp_2026-01.csv"
if not PRECIP_CSV.exists():
    PRECIP_CSV = DEFAULT_FORCING_DIR / "precip_2026-01.csv"
if not TEMP_CSV.exists():
    TEMP_CSV = DEFAULT_FORCING_DIR / "temp_2026-01.csv"

NUMBER_OF_ITERATIONS = 1000
USE_SINK_FILL = True
IGNORE_SINK_OVERFILL = False

DEM_NODATA = -999999.0
FIELD_NODATA = {
    "soil__thickness": 2.549999999999999822e+00,
    "soil__density": 4.259999999999999898e-01,
    "soil__internal_friction_angle": 2.700000000000000000e+01,
    "soil__saturated_hydraulic_conductivity": 7.052563416309979523e+93,
    "vegetation__plant_functional_type": -9.999000000000000000e+03,
    "soil__maximum_total_cohesion": -9.999000000000000000e+03,
    "soil__mode_total_cohesion": -9.999000000000000000e+03,
    "soil__minimum_total_cohesion": -9.999000000000000000e+03,
}

LAI_BY_PFT = {0: 1.5, 1: 2.0, 2: 4.0, 3: 1.0}
COHESION_FIELDS = (
    "soil__minimum_total_cohesion",
    "soil__mode_total_cohesion",
    "soil__maximum_total_cohesion",
)

LATITUDE = 47.7
ALBEDO = 0.2
ZVEG = 0.5
Z_WIND = 2.0
VWIND = 3.04
RELATIVE_HUMIDITY = 0.8
LAPSE_RATE_C_PER_KM = 4.5
INITIAL_TIME_YEARS = 0.22
STORM_DURATION_HOURS = 24.0

MIN_SOIL_THICKNESS_M = 0.01
MIN_TRANSMISSIVITY = 0.01
RECHARGE_FLOOR = 0.01

APPLY_COHESION_REDUCTION = False
COHESION_REDUCTION_BY_BURN = {1: 0.00, 2: 0.15, 3: 0.35, 4: 0.60}


In [ ]:
def add_or_update_field(grid, name, values, at="node"):
    values = np.asarray(values)
    container = grid.at_node if at == "node" else grid.at_cell
    if name in container:
        container[name][:] = values
    else:
        grid.add_field(name, values.copy(), at=at, clobber=True)
    return container[name]


def load_ascii_values(path, field_name):
    with open(path) as f:
        temp_grid = esri_ascii.load(f, name=field_name)
    return temp_grid.at_node[field_name]


def load_node_field(grid, base_dir, filename, field_name, nodata_value=None, transform=None, dtype=None):
    raw = load_ascii_values(base_dir / filename, field_name)
    values = raw.copy()
    if transform is not None:
        values = transform(values)
    if dtype is not None:
        values = values.astype(dtype)
    add_or_update_field(grid, field_name, values, at="node")
    if nodata_value is not None:
        grid.set_nodata_nodes_to_closed(raw, nodata_value)
    return raw, grid.at_node[field_name]


def close_dem_nodata(grid, nodata_value=DEM_NODATA):
    z = grid.at_node["topographic__elevation"]
    bad = np.isclose(z, nodata_value, atol=1e-2) | (z < -1e5)
    grid.status_at_node[bad] = grid.BC_NODE_IS_CLOSED
    z[bad] = 0.0
    return z


def load_forcing_arrays(precip_csv, temp_csv, elevation, z_ref, lapse_rate):
    precip = pd.read_csv(precip_csv, parse_dates=["datetime"]).sort_values("datetime")
    temp = pd.read_csv(temp_csv, parse_dates=["datetime"]).sort_values("datetime")
    forcing = precip.merge(temp, on="datetime", how="inner", validate="one_to_one")
    if forcing.empty:
        raise ValueError("No forcing rows were loaded")

    delta_z_km = (elevation - z_ref) / 1000.0
    rainfall_arrays = [np.full_like(elevation, p, dtype=float) for p in forcing["precip_mm"].to_numpy()]
    tempmin_arrays = [t - lapse_rate * delta_z_km for t in forcing["tmin_c"].to_numpy()]
    tempmax_arrays = [t - lapse_rate * delta_z_km for t in forcing["tmax_c"].to_numpy()]
    return forcing, rainfall_arrays, tempmin_arrays, tempmax_arrays


def cell_to_node(grid, cell_values, fill_value=0.0):
    out = np.full(grid.number_of_nodes, fill_value, dtype=float)
    out[grid.node_at_cell] = np.asarray(cell_values)
    return out


def patch_component_field_overwrite():
    def _overwrite_process_field(self, field, field_name):
        if isinstance(field, np.ndarray) and np.shape(field) == np.shape(self._grid.at_node["topographic__elevation"]):
            if field_name in self._gridCopy.at_node:
                self._gridCopy.at_node[field_name][:] = field
            else:
                self._gridCopy.add_field(field_name, field.copy(), at="node")
            return map_node_to_cell(self._gridCopy, field_name)
        return field

    Radiation._process_field = _overwrite_process_field
    PotentialEvapotranspiration._process_field = _overwrite_process_field


def update_cohesion_fields(grid, reduction_by_burn=None):
    burn = grid.at_node["burn__severity"].astype(int)
    mult = np.ones(grid.number_of_nodes, dtype=float)
    if reduction_by_burn is not None:
        for cls, red in reduction_by_burn.items():
            mult[burn == cls] = 1.0 - red

    for field in COHESION_FIELDS:
        backup_field = f"{field}_pre"
        if backup_field not in grid.at_node:
            add_or_update_field(grid, backup_field, grid.at_node[field].copy(), at="node")
        values = grid.at_node[backup_field].copy() * mult
        add_or_update_field(grid, field, values, at="node")


## Terrain Grid

Load the DEM, close DEM nodata robustly, set the outlet using the original watershed-style flow, and compute routing products needed downstream.

In [ ]:
with open(SCENARIO_DIR / "topographic__elevation.asc") as f:
    grid = esri_ascii.load(f, name="topographic__elevation")

Z = close_dem_nodata(grid, nodata_value=DEM_NODATA)
outlet_id = grid.core_nodes[np.argmin(Z[grid.core_nodes])]
grid.set_watershed_boundary_condition_outlet_id(outlet_id, Z)

if USE_SINK_FILL:
    sink_filler = SinkFillerBarnes(
        grid,
        "topographic__elevation",
        method="D8",
        fill_flat=False,
        ignore_overfill=IGNORE_SINK_OVERFILL,
    )
    sink_filler.run_one_step()

flow_accumulator = FlowAccumulator(
    grid,
    surface="topographic__elevation",
    flow_director="FlowDirectorD8",
    runoff_rate=None,
)
drainage_area, discharge = flow_accumulator.accumulate_flow()

add_or_update_field(grid, "topographic__slope", grid.at_node["topographic__steepest_slope"], at="node")
add_or_update_field(
    grid,
    "topographic__specific_contributing_area",
    drainage_area / grid.dx,
    at="node",
)

Zmin = float(Z[grid.core_nodes].min())
Zmax = float(Z[grid.core_nodes].max())


## Static Inputs

Load all scenario rasters, preserve variable-specific nodata closure values, and derive the fields required by the ecohydrology and landslide components.

In [ ]:
load_node_field(
    grid,
    SCENARIO_DIR,
    "soil__thickness.asc",
    "soil__thickness",
    nodata_value=FIELD_NODATA["soil__thickness"],
    transform=lambda x: x / 100.0,
)
load_node_field(
    grid,
    SCENARIO_DIR,
    "soil__density.asc",
    "soil__density",
    nodata_value=FIELD_NODATA["soil__density"],
)
load_node_field(
    grid,
    SCENARIO_DIR,
    "soil__internal_friction_angle.asc",
    "soil__internal_friction_angle",
    nodata_value=FIELD_NODATA["soil__internal_friction_angle"],
)
load_node_field(grid, SCENARIO_DIR, "porosity.asc", "porosity")
load_node_field(grid, SCENARIO_DIR, "field__capacity.asc", "field__capacity")
load_node_field(grid, SCENARIO_DIR, "wilting__point.asc", "wilting__point")
load_node_field(
    grid,
    SCENARIO_DIR,
    "soil__saturated_hydraulic_conductivity.asc",
    "soil__saturated_hydraulic_conductivity",
    nodata_value=FIELD_NODATA["soil__saturated_hydraulic_conductivity"],
)
load_node_field(
    grid,
    SCENARIO_DIR,
    "vegetation__plant_functional_type.asc",
    "vegetation__plant_functional_type",
    nodata_value=FIELD_NODATA["vegetation__plant_functional_type"],
    dtype=int,
)
load_node_field(
    grid,
    SCENARIO_DIR,
    "soil__maximum_total_cohesion.asc",
    "soil__maximum_total_cohesion",
    nodata_value=FIELD_NODATA["soil__maximum_total_cohesion"],
)
load_node_field(
    grid,
    SCENARIO_DIR,
    "soil__mode_total_cohesion.asc",
    "soil__mode_total_cohesion",
    nodata_value=FIELD_NODATA["soil__mode_total_cohesion"],
)
load_node_field(
    grid,
    SCENARIO_DIR,
    "soil__minimum_total_cohesion.asc",
    "soil__minimum_total_cohesion",
    nodata_value=FIELD_NODATA["soil__minimum_total_cohesion"],
)
load_node_field(grid, SCENARIO_DIR, "burn__severity.asc", "burn__severity")

open_nodes = grid.status_at_node != grid.BC_NODE_IS_CLOSED
hs = grid.at_node["soil__thickness"]
hs[(open_nodes) & (hs <= 0)] = MIN_SOIL_THICKNESS_M

ksat = grid.at_node["soil__saturated_hydraulic_conductivity"]
transmissivity = ksat * 2.5 * hs
transmissivity[transmissivity <= 0] = MIN_TRANSMISSIVITY
add_or_update_field(grid, "soil__transmissivity", transmissivity, at="node")

pft = grid.at_node["vegetation__plant_functional_type"].astype(int)
pft[pft < 0] = 0
grid.at_node["vegetation__plant_functional_type"][:] = pft
add_or_update_field(
    grid,
    "vegetation__plant_functional_type",
    pft[grid.node_at_cell].astype(int),
    at="cell",
)

lai = np.full(grid.number_of_nodes, LAI_BY_PFT[3], dtype=float)
for cls, value in LAI_BY_PFT.items():
    lai[pft == cls] = value
add_or_update_field(grid, "vegetation__live_leaf_area_index", lai, at="node")
add_or_update_field(grid, "vegetation__cover_fraction", lai / 4.0, at="node")
add_or_update_field(grid, "vegetation__live_leaf_area_index", lai[grid.node_at_cell], at="cell")
add_or_update_field(grid, "vegetation__cover_fraction", (lai / 4.0)[grid.node_at_cell], at="cell")

burn = grid.at_node["burn__severity"].astype(int)
burn[~np.isin(burn, [2, 3, 4])] = 1
grid.at_node["burn__severity"][:] = burn

initial_saturation = (
    0.5 * (grid.at_node["field__capacity"] - grid.at_node["wilting__point"])
    + grid.at_node["wilting__point"]
) / grid.at_node["porosity"]
add_or_update_field(
    grid,
    "soil_moisture__initial_saturation_fraction",
    initial_saturation,
    at="node",
)
add_or_update_field(
    grid,
    "soil_moisture__initial_saturation_fraction",
    initial_saturation[grid.node_at_cell],
    at="cell",
)
add_or_update_field(
    grid,
    "saturated__hydraulic_conductivity",
    ksat[grid.node_at_cell],
    at="cell",
)
add_or_update_field(
    grid,
    "rainfall__daily_depth",
    np.zeros(grid.number_of_cells, dtype=float),
    at="cell",
)

update_cohesion_fields(
    grid,
    COHESION_REDUCTION_BY_BURN if APPLY_COHESION_REDUCTION else None,
)


## Ecohydrology

Load forcing, reuse PET and soil-moisture components across all days, and aggregate runoff/recharge outputs for the landslide step.

In [ ]:
forcing, rainfall_arrays, tempmin_arrays, tempmax_arrays = load_forcing_arrays(
    PRECIP_CSV,
    TEMP_CSV,
    Z,
    Zmin,
    LAPSE_RATE_C_PER_KM,
)

patch_component_field_overwrite()
pet = PotentialEvapotranspiration(grid, method="PenmanMonteith")
soil_moisture = SoilMoisture(grid)

pet._latitude = LATITUDE
pet._a = ALBEDO
pet._zm = Z_WIND
pet._zveg = ZVEG * np.ones(grid.number_of_cells)
pet._vz = VWIND * np.ones(grid.number_of_cells)
pet._relative_humidity = RELATIVE_HUMIDITY * np.ones(grid.number_of_cells)
pet._LAI = grid.at_node["vegetation__live_leaf_area_index"]

soil_moisture._Tb = STORM_DURATION_HOURS

node_at_cell = grid.node_at_cell
current_time = INITIAL_TIME_YEARS

runoff_arrays = []
recharge_arrays = []
soil_moisture_arrays = []
ET_arrays = []

for rainfall, tempmin, tempmax in zip(rainfall_arrays, tempmin_arrays, tempmax_arrays):
    add_or_update_field(grid, "Precipitation", rainfall, at="node")
    add_or_update_field(grid, "Tmin", tempmin, at="node")
    add_or_update_field(grid, "Tmax", tempmax, at="node")
    grid.at_cell["rainfall__daily_depth"][:] = rainfall[node_at_cell]

    pet._current_time = current_time
    pet._Tmin = tempmin
    pet._Tmax = tempmax
    pet.update()

    soil_moisture._current_time = current_time
    soil_moisture.update()

    recharge_arrays.append(cell_to_node(grid, grid.at_cell["soil_moisture__root_zone_leakage"]))
    runoff_arrays.append(cell_to_node(grid, grid.at_cell["surface__runoff"]))
    soil_moisture_arrays.append(cell_to_node(grid, grid.at_cell["soil_moisture__saturation_fraction"]))
    ET_arrays.append(cell_to_node(grid, grid.at_cell["surface__evapotranspiration"]))

    current_time = soil_moisture.current_time

mean_runoff = np.mean(runoff_arrays, axis=0)
mean_recharge = np.mean(recharge_arrays, axis=0)
max_runoff = np.maximum.reduce(runoff_arrays)
max_recharge = np.maximum.reduce(recharge_arrays)


## Landslide Probability

Build the recharge inputs and run `LandslideProbability` using the same field names expected by Landlab.

In [ ]:
r = max_recharge.copy()
r[r <= 0] = RECHARGE_FLOOR
rstd = r * 0.1

add_or_update_field(grid, "groundwater__recharge_mean", r, at="node")
add_or_update_field(grid, "groundwater__recharge_standard_deviation", rstd, at="node")
add_or_update_field(grid, "groundwater__runoff_mean", max_runoff, at="node")
add_or_update_field(grid, "test_runoff", mean_runoff, at="node")
add_or_update_field(grid, "test_recharge", mean_recharge, at="node")

LS_prob = LandslideProbability(
    grid,
    number_of_iterations=NUMBER_OF_ITERATIONS,
    groundwater__recharge_distribution="lognormal_spatial",
    groundwater__recharge_mean=r,
    groundwater__recharge_standard_deviation=rstd,
)
LS_prob.calculate_landslide_probability()

results = {
    "forcing": forcing,
    "mean_runoff": mean_runoff,
    "mean_recharge": mean_recharge,
    "max_runoff": max_runoff,
    "max_recharge": max_recharge,
    "runoff_arrays": runoff_arrays,
    "recharge_arrays": recharge_arrays,
    "soil_moisture_arrays": soil_moisture_arrays,
    "ET_arrays": ET_arrays,
}

print(
    f"Pipeline complete for {len(forcing)} forcing days, {grid.number_of_core_nodes} core nodes, and {NUMBER_OF_ITERATIONS} landslide iterations."
)
